# PBMC-10k 复现实验

PBMC 10k RNA-ATAC 多模态单细胞数据。本 notebook 默认使用 `two_group` 协议，并将结果写入发布包的 `outputs/PBMC-10k/`。

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

def find_project_root(start):
    for path in (start, *start.parents):
        if (path / 'model' / 'main_dpcl.py').is_file() and (path / 'scDPCL_release').is_dir():
            return path
    raise FileNotFoundError('无法定位 scMDCL-main 项目根目录')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
RELEASE_ROOT = PROJECT_ROOT / 'scDPCL_release'
os.chdir(PROJECT_ROOT)
DATASET = 'PBMC-10k'
PROFILE = 'two_group'
print('Python:', sys.executable)
print('Project:', PROJECT_ROOT)
print('Dataset:', DATASET)


In [ ]:
manifest = json.loads((RELEASE_ROOT / 'config' / 'datasets.json').read_text(encoding='utf-8'))
info = manifest[DATASET]
print(f"cells={info['cells']:,}, clusters={info['clusters']}")
print(f"RNA dim={info['rna_dim']}, {info['second_view']} dim={info['second_dim']}")
print('available k:', info['available_k'])
print('two-group reference ARI:', info['two_group_best_ari'])
print('active clusters:', info['two_group_active_clusters'])


## 1. 输入检查

下面只检查数据、权重和完整命令，不启动训练。

In [ ]:
dry_command = [
    sys.executable, str(RELEASE_ROOT / 'run.py'),
    '--dataset', DATASET,
    '--profile', PROFILE,
    '--dry-run',
]
subprocess.run(dry_command, cwd=PROJECT_ROOT, check=True)


## 2. 正式训练

将 `RUN_TRAINING` 改为 `True` 后运行。训练日志、命令与汇总指标会保存在独立输出目录。

In [ ]:
RUN_TRAINING = False
OUTPUT_ROOT = RELEASE_ROOT / 'outputs' / DATASET
train_command = [
    sys.executable, str(RELEASE_ROOT / 'run.py'),
    '--dataset', DATASET,
    '--profile', PROFILE,
    '--output-root', str(OUTPUT_ROOT),
]

if RUN_TRAINING:
    subprocess.run(train_command, cwd=PROJECT_ROOT, check=True)
else:
    print('训练未启动。确认 dry-run 后，将 RUN_TRAINING 设置为 True。')
    print('命令:', subprocess.list2cmdline(train_command))


## 3. 读取结果

训练完成后运行此单元，显示 `summary.json` 中的全部指标。

In [ ]:
summary_path = OUTPUT_ROOT / 'summary.json'
if summary_path.is_file():
    rows = json.loads(summary_path.read_text(encoding='utf-8'))
    for row in rows:
        print(json.dumps(row, ensure_ascii=False, indent=2))
else:
    print('尚无本次 notebook 输出:', summary_path)
